# Numbers quoted in the report

Reproduce the numbers of report 02 that the outputs of
[05_01](05_01_calibrate.ipynb), [05_02](05_02_run_vlm.ipynb) and
[05_03](05_03_accuracy_latency.ipynb) do not show.
Nothing is measured here: every number comes from files those notebooks read or wrote.

In [1]:
from math        import ceil
from pathlib     import Path
from scipy.stats import bootstrap

from huggingface_hub import hf_hub_download

import json

import numpy  as np
import pandas as pd

/home/yabsed/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Projections

Every block of the language model has seven projections.
On flash, projection $p$ is $N_p$ rows of $D_p$ weights each, one row per input channel.
The configuration of the model gives both.

In [2]:
MODEL_ID       = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
MODEL_REVISION = "74dd0bf867a4cda7950c17663794267c60cf4b40"

In [3]:
config_file = hf_hub_download(
    MODEL_ID,
    "config.json",
    revision = MODEL_REVISION,
)

# Configuration of the language model
config = json.loads(Path(config_file).read_text())["text_config"]

In [4]:
hidden = config["hidden_size"]        # Width of the hidden state
wide   = config["intermediate_size"]  # Width inside the MLP

# Keys and values have fewer heads than queries.
head   = hidden // config["num_attention_heads"]
narrow = head * config["num_key_value_heads"]

hidden, wide, narrow

(896, 4864, 128)

In [5]:
# projection -> (N, D)
shapes = {
    "q_proj"    : (hidden, hidden),
    "k_proj"    : (hidden, narrow),
    "v_proj"    : (hidden, narrow),
    "o_proj"    : (hidden, hidden),
    "gate_proj" : (hidden, wide),
    "up_proj"   : (hidden, wide),
    "down_proj" : (wide,   hidden),
}

projections = pd.DataFrame(shapes, index=["N", "D"]).T

A weight takes two bytes, so one row is $w = 2D/1024$ KiB.

In [6]:
projections["row_kib"] = 2 * projections.D / 1024

projections

,N,D,row_kib
q_proj,896,896,1.75
k_proj,896,128,0.25
v_proj,896,128,0.25
o_proj,896,896,1.75
gate_proj,896,4864,9.50
up_proj,896,4864,9.50
down_proj,4864,896,1.75


## 2. Search steps

One step of the search in 05_01 lowers the fraction of one projection by

$$
\delta_p = \alpha \, \frac{N_q D_q}{N_p D_p},
$$

where $q$ is `q_proj` and $\alpha = 0.05$.

In [7]:
BASE_STEP = 0.05

# Weights of each projection
weights = projections.N * projections.D

steps = pd.DataFrame({
    "step" : BASE_STEP * weights["q_proj"] / weights,
})

# Rows that one step takes off the projection
steps["rows"] = steps.step * projections.N

steps.round(4)

,step,rows
q_proj,0.0500,44.8000
k_proj,0.3500,313.6000
v_proj,0.3500,313.6000
o_proj,0.0500,44.8000
gate_proj,0.0092,8.2526
up_proj,0.0092,8.2526
down_proj,0.0092,44.8000


Whichever projection takes the step, the block loses the same number of weights:

In [8]:
step_weights  = BASE_STEP * weights["q_proj"]
block_weights = weights.sum()

# Weights of one step, weights of a block, and the first as a percentage of the second
step_weights, block_weights, round(100 * step_weights / block_weights, 3)

(np.float64(40140.8), np.int64(14909440), np.float64(0.269))

## 3. Rows per projection

`budgets.csv` holds the rows $R$ of every projection at every requested fraction $f$.
05_02 ran at three of them.

In [9]:
FRACTIONS = [0.3, 0.5, 0.7]

In [10]:
budgets = pd.read_csv("budgets.csv")

# layers.0.self_attn.q_proj -> q_proj
budgets["kind"] = budgets.projection.str.split(".").str[-1]

budgets.projection.nunique()

168

In [11]:
# At f = 1 every projection reads all of its rows.
full = budgets[budgets.fraction == 1.0]

assert (full.R == full.kind.map(projections.N)).all()

The percentage of its rows that a projection reads, averaged over the blocks:

In [12]:
budgets["share"] = 100 * budgets.R / budgets.kind.map(projections.N)

shares = budgets.pivot_table(
    index   = "fraction",
    columns = "kind",
    values  = "share",
    aggfunc = "mean",
)

shares.loc[FRACTIONS, projections.index].round(1)

,q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj
fraction,,,,,,,
0.3,22.6,47.4,76.6,34.5,25.7,40.1,22.0
0.5,26.8,59.1,81.0,41.6,47.4,66.0,41.0
0.7,36.4,66.4,86.9,51.4,68.7,86.6,63.3


The percentage of all projection weights that the model reads:

In [13]:
budgets["weights"] = budgets.R * budgets.kind.map(projections.D)

read = budgets.groupby("fraction").weights.sum()

(100 * read / read[1.0]).loc[FRACTIONS].round(1)

fraction
0.3    29.7
0.5    49.9
0.7    69.8
Name: weights, dtype: float64

## 4. Tile lengths

05_02 takes the median latency of each profiled read size from
[01](../01/01_01_measure_chunk_latency.ipynb).
$z_*$ is the size with the lowest cost per KiB, and Tiles use $B = \lceil z_* / w \rceil$ rows.

In [14]:
measurements = pd.read_csv("../01/chunk_latency_measurements.csv")

latencies = (
    measurements.groupby("size_kib")["latency_ms"]
    .median()
    .sort_index()
)

tile_reference_kib = float((latencies / latencies.index).idxmin())
tile_reference_kib

648.0

In [15]:
tile_lengths = []

for row_kib in projections.row_kib:
    tile_lengths.append(ceil(tile_reference_kib / row_kib))

projections["B"] = tile_lengths

projections

,N,D,row_kib,B
q_proj,896,896,1.75,371
k_proj,896,128,0.25,2592
v_proj,896,128,0.25,2592
o_proj,896,896,1.75,371
gate_proj,896,4864,9.50,69
up_proj,896,4864,9.50,69
down_proj,4864,896,1.75,371


## 5. Accuracy differences

Every condition answered the same questions.
Pair the answers by question and take the difference in accuracy between two conditions,
in percentage points, with the 95% BCa bootstrap interval of 05_03.

In [16]:
answers = pd.read_csv("answers.csv")

# scores[question, (method, fraction)]: 100 for a right answer, 0 for a wrong one
scores = 100.0 * answers.pivot(
    index   = "question_id",
    columns = ["method", "fraction"],
    values  = "correct",
)

len(scores)

391

In [17]:
def interval(values, statistic):
    """Return the 95% BCa bootstrap interval of a statistic: (low, high)."""

    result = bootstrap(
        (values,),
        statistic,
        confidence_level = 0.95,
        n_resamples      = 10_000,
        method           = "BCa",
        rng              = 0,
    )

    return result.confidence_interval.low, result.confidence_interval.high

In [18]:
def difference(first, second):
    """Return the accuracy of one condition minus that of another: (mean, low, high)."""

    values    = (scores[first] - scores[second]).to_numpy()
    low, high = interval(values, np.mean)

    return values.mean(), low, high

Between two methods at the same $f$:

In [19]:
PAIRS = [
    ("Tiles",        "Paper greedy"),
    ("Tiles",        "Top-R"),
    ("Paper greedy", "Top-R"),
]

records = []

for first, second in PAIRS:
    for fraction in FRACTIONS:
        mean, low, high = difference((first, fraction), (second, fraction))

        records.append({
            "pair"       : f"{first} - {second}",
            "fraction"   : fraction,
            "difference" : mean,
            "low"        : low,
            "high"       : high,
        })

between = pd.DataFrame(records)
between = between.set_index(["pair", "fraction"])

between.round(1)

difference  low  high
pair                 fraction                       
Tiles - Paper greedy 0.3              0.0 -2.3   2.3
                     0.5             -0.3 -2.8   2.3
                     0.7              0.5 -1.0   2.3
Tiles - Top-R        0.3             -1.0 -4.1   2.0
                     0.5             -0.3 -3.1   2.6
                     0.7             -0.3 -1.5   1.0
Paper greedy - Top-R 0.3             -1.0 -3.8   1.8
                     0.5              0.0 -2.6   2.6
                     0.7             -0.8 -2.8   1.0

The width of each interval:

In [20]:
widths = between.high - between.low

widths.unstack("fraction").round(1)

fraction,0.3,0.5,0.7
pair,,,
Paper greedy - Top-R,5.6,5.1,3.8
Tiles - Paper greedy,4.6,5.1,3.3
Tiles - Top-R,6.1,5.6,2.6


Between a method and `Dense`:

In [21]:
METHODS = ["Top-R", "Paper greedy", "Tiles"]

records = []

for method in METHODS:
    for fraction in FRACTIONS:
        mean, low, high = difference((method, fraction), ("Dense", 1.0))

        records.append({
            "method"     : method,
            "fraction"   : fraction,
            "difference" : mean,
            "low"        : low,
            "high"       : high,
        })

against_dense = pd.DataFrame(records)
against_dense = against_dense.set_index(["method", "fraction"])

against_dense.round(1)

difference   low  high
method       fraction                        
Top-R        0.3             -4.9  -8.7  -1.3
             0.5             -2.6  -5.9   0.8
             0.7             -0.5  -3.1   2.0
Paper greedy 0.3             -5.9  -9.7  -2.0
             0.5             -2.6  -6.1   0.8
             0.7             -1.3  -4.1   1.3
Tiles        0.3             -5.9 -10.0  -1.8
             0.5             -2.8  -6.1   0.3
             0.7             -0.8  -3.6   1.8

## 6. Times relative to Dense

As in 05_03, `load_ms` is the selection time plus the read time of a frame,
and a condition is summarized by the median over its recorded frames.

In [22]:
frame_times = pd.read_csv("frame_times.csv")

frame_times["load_ms"] = frame_times.selection_ms + frame_times.io_ms

In [23]:
TIMES = ["selection_ms", "io_ms", "load_ms"]

medians = frame_times.groupby(["method", "fraction"], sort=False)[TIMES].median()

Each median divided by that of `Dense`:

In [24]:
dense = medians.loc[("Dense", 1.0)]

(medians / dense).round(2)

selection_ms  io_ms  load_ms
method       fraction                              
Dense        1.0               1.00   1.00     1.00
Top-R        0.3               9.33   3.29     3.39
             0.5               9.37   3.97     4.06
             0.7               9.43   3.84     3.93
Paper greedy 0.3              22.76   0.42     0.78
             0.5              23.61   0.60     0.97
             0.7              24.59   0.76     1.16
Tiles        0.3               1.59   0.42     0.44
             0.5               1.58   0.60     0.61
             0.7               1.57   0.75     0.76

The selection plus read time of `Tiles` divided by that of `Paper greedy`:

In [25]:
(medians.loc["Tiles"].load_ms / medians.loc["Paper greedy"].load_ms).round(2)

fraction
0.3    0.56
0.5    0.63
0.7    0.66
Name: load_ms, dtype: float64